In [0]:
-- 1. Active CineClub members on the latest snapshot (BR-022)
CREATE OR REPLACE TEMP VIEW dw_active_members AS
SELECT
    cde_id,
    MIN(plan_id) AS plan_id                -- 1 row per cde_id; see check 1
FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
WHERE snapshot_date_id = (SELECT MAX(snapshot_date_id)
                          FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot)
  AND renewal_status LIKE 'Active%'
  AND cde_id > 0
GROUP BY cde_id;

-- 2. Net Indian-film tickets per CDE_ID x title, P24M (refunds netted)
CREATE OR REPLACE TEMP VIEW dw_indian_titles AS
SELECT
    t.CDE_ID,
    f.title_id,
    SUM(t.Quantity)                                               AS net_tickets,
    SUM(CASE WHEN t.SCENEFLAG = 'Yes' THEN t.Quantity ELSE 0 END) AS net_scene_tickets
FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
JOIN cpx_dataanalytics_gold.customer360.cpx_d_film f
  ON t.film_id = f.film_id
WHERE t.TicketFLAG = 'Yes'
  AND f.language IN ('Hindi','Punjabi','Gujarati','Telugu','Tamil')
  AND t.VisitDateId >= CAST(date_format(add_months(current_date(), -24), 'yyyyMMdd') AS INT)
  AND t.VisitDateId <  CAST(date_format(current_date(), 'yyyyMMdd') AS INT)
  AND t.CDE_ID > 0
GROUP BY t.CDE_ID, f.title_id;

-- 3. Audience (both cohorts) + membership status + distinct Indian titles
CREATE OR REPLACE TEMP VIEW dw_audience AS
WITH viewers AS (
    SELECT
        CDE_ID,
        COUNT_IF(net_tickets > 0)                           AS indian_titles_p24m,
        COUNT_IF(net_tickets > 0 AND net_scene_tickets > 0) AS indian_titles_scene_p24m
    FROM dw_indian_titles
    GROUP BY CDE_ID
)
SELECT
    v.CDE_ID,
    CASE
        WHEN a.cde_id IS NULL        THEN 'Non-member'
        WHEN a.plan_id = 'annual1'   THEN 'Annual'
        WHEN a.plan_id = 'monthly1'  THEN 'Monthly'
        WHEN a.plan_id = 'gift-plan' THEN 'Gift Plan'
        ELSE a.plan_id
    END AS membership_status,
    v.indian_titles_p24m
FROM viewers v
LEFT JOIN dw_active_members a
  ON v.CDE_ID = a.cde_id
WHERE (a.cde_id IS NOT NULL AND v.indian_titles_p24m > 0)         -- CineClub cohort
   OR (a.cde_id IS NULL     AND v.indian_titles_scene_p24m > 0);  -- Scene+ non-CC cohort

-- 4. Final output: one row per CDE_ID
CREATE OR REPLACE TEMP VIEW dw_final AS
WITH spend_p12m AS (
    SELECT
        t.CDE_ID,
        SUM(t.Net_Revenue) AS total_spend_p12m
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN dw_audience p
      ON t.CDE_ID = p.CDE_ID
    WHERE t.Transaction_DateTime >= add_months(current_date(), -12)
      AND t.Transaction_DateTime <  current_date()
      -- AND t.RecRoomFLAG = 'No' AND t.STOREFLAG = 'No'   -- uncomment for theatre-only spend
    GROUP BY t.CDE_ID
),
gift_purchasers AS (
    SELECT DISTINCT c.cde_id
    FROM cpx_dataanalytics_silver.cineclub.gifting g
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_customer c
      ON g.purchaser_id = c.cc_user_profile_id
    WHERE c.cde_id > 0
)
SELECT
    p.CDE_ID,
    p.membership_status,
    p.indian_titles_p24m,
    ROUND(COALESCE(s.total_spend_p12m, 0), 2)               AS total_spend_p12m,
    CASE WHEN g.cde_id IS NOT NULL THEN 'Yes' ELSE 'No' END AS gift_purchased_ever
FROM dw_audience p
LEFT JOIN spend_p12m s
  ON p.CDE_ID = s.CDE_ID
LEFT JOIN gift_purchasers g
  ON p.CDE_ID = g.cde_id;

SELECT * FROM dw_final;

-- 5. Summary by status (use this to reconcile)
SELECT
    membership_status,
    COUNT(*)                                                      AS customers,
    MIN(indian_titles_p24m)                                       AS min_titles,
    ROUND(AVG(indian_titles_p24m), 2)                             AS avg_titles,
    SUM(CASE WHEN total_spend_p12m = 0 THEN 1 ELSE 0 END)         AS zero_spend,
    SUM(CASE WHEN total_spend_p12m < 0 THEN 1 ELSE 0 END)         AS negative_spend,
    ROUND(AVG(total_spend_p12m), 2)                               AS avg_spend_p12m,
    SUM(CASE WHEN gift_purchased_ever = 'Yes' THEN 1 ELSE 0 END)  AS gift_purchasers
FROM dw_final
GROUP BY membership_status
ORDER BY membership_status;


-- 6. Summary by status (use this to reconcile)
SELECT COUNT(*) AS non_members_zero_titles
FROM dw_final
WHERE indian_titles_p24m = 0;